# Task 9: Create Backend
Setup Flask backend to handle user requests for prediction.

In [ ]:
backend_code = '''
from flask import Flask, render_template, request, jsonify
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

app = Flask(__name__)

scaler = joblib.load("scaler.pkl")

MODEL_MAP = {
    "linear_regression": {"file": "model_linear_regression.pkl", "name": "Linear Regression", "is_regressor": True},
    "logistic_regression": {"file": "model_logistic_regression.pkl", "name": "Logistic Regression", "is_regressor": False},
    "svm": {"file": "model_svm.pkl", "name": "SVM", "is_regressor": False},
    "knn": {"file": "model_knn.pkl", "name": "KNN", "is_regressor": False},
    "decision_tree": {"file": "model_decision_tree.pkl", "name": "Decision Tree", "is_regressor": False},
    "random_forest": {"file": "model_random_forest.pkl", "name": "Random Forest", "is_regressor": False},
}

models = {}
for key, info in MODEL_MAP.items():
    try:
        models[key] = joblib.load(info["file"])
    except Exception as e:
        print(f"Warning: Could not load {info['file']}: {e}")


def prepare_features(data):
    age_years = data["age"]
    height = data["height"]
    weight = data["weight"]
    bmi = round(weight / ((height / 100) ** 2), 2)
    pulse_pressure = data["ap_hi"] - data["ap_lo"]

    features = np.array([[
        data["gender"], height, weight,
        data["ap_hi"], data["ap_lo"],
        data["cholesterol"], data["gluc"],
        data["smoke"], data["alco"], data["active"],
        age_years, bmi, pulse_pressure
    ]])
    return scaler.transform(features)


def get_prediction(model_key, features):
    model = models[model_key]
    info = MODEL_MAP[model_key]

    if info["is_regressor"]:
        raw = model.predict(features)[0]
        pred = 1 if raw >= 0.5 else 0
        conf = raw if pred == 1 else 1 - raw
    elif hasattr(model, "predict_proba"):
        proba = model.predict_proba(features)[0]
        pred = int(np.argmax(proba))
        conf = float(proba[pred])
    else:
        pred = int(model.predict(features)[0])
        conf = 0.75

    return {"model": info["name"], "prediction": pred, "confidence": round(conf, 4)}


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    features = prepare_features(data)
    model_choice = data.get("model", "all")

    if model_choice == "all":
        predictions = [get_prediction(key, features) for key in models]
    else:
        predictions = [get_prediction(model_choice, features)]

    return jsonify({"predictions": predictions})


@app.route("/comparison")
def comparison():
    df = pd.read_csv("cardio_cleaned.csv")
    X = df.drop("cardio", axis=1)
    y = df["cardio"]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_test_scaled = scaler.transform(X_test)

    results = []
    for key, model in models.items():
        info = MODEL_MAP[key]
        if info["is_regressor"]:
            y_pred = (model.predict(X_test_scaled) >= 0.5).astype(int)
        else:
            y_pred = model.predict(X_test_scaled)

        results.append({
            "model": info["name"],
            "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
            "precision": round(float(precision_score(y_test, y_pred)), 4),
            "recall": round(float(recall_score(y_test, y_pred)), 4),
            "f1_score": round(float(f1_score(y_test, y_pred)), 4),
        })

    return jsonify(results)


if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(backend_code.strip())

print("app.py created.")
print("\nRun the server with: python app.py")
print("Open: http://localhost:5000")

## 9.1 Backend Architecture

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/` | GET | Serve the frontend page |
| `/predict` | POST | Accept patient data, return predictions |
| `/comparison` | GET | Return accuracy metrics for all models |

### Request Flow:
1. User fills in patient data on the form
2. JavaScript sends POST request with JSON to `/predict`
3. Flask receives data, engineers features (BMI, pulse pressure, age)
4. Features are scaled using the saved `StandardScaler`
5. Selected model(s) make predictions
6. Response with prediction + confidence is returned as JSON
7. JavaScript renders the results dynamically

In [ ]:
import os

required_files = ['scaler.pkl', 'app.py', 'templates/index.html',
                  'static/css/style.css', 'static/js/app.js']

model_files = [f'model_{m}.pkl' for m in
               ['linear_regression', 'logistic_regression', 'svm', 'knn', 'decision_tree', 'random_forest']]

print("File Check:")
for f in required_files + model_files:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f"  {status}  {f}")